### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output. 

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="groq:qwen/qwen3-32b",
)

### Different data types 

#### Pydantic

 Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [13]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating of the movie")
    director: str = Field(description="The director of the movie")


In [14]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11a2cbfd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11a2f4070>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'rating': {'description': 'The rating of the movie', 'type': 'number'}, 'director': {'description': 'The director of the movie', 'type': 'str

In [ ]:
# Unstructured output

model.invoke("Provide the details about the movie Inception")

AIMessage(content='<think>\nOkay, so the user wants details about the movie Inception. Let me start by recalling what I know about it. Inception is a sci-fi action film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, right? He plays Dom Cobb, a thief who steals information by infiltrating the subconscious. The antagonist is Mal (Marion Cotillard), who\'s his wife. The movie\'s about dreams and entering people\'s dreams to steal secrets or plant ideas.\n\nI should mention the plot structure. It\'s non-linear, with different layers of dreams and reality. The concept of a totem, like the spinning top, which they use to determine if they\'re dreaming. The team includes Arthur (Joseph Gordon-Levitt), Ariadne (Elliot Page), Eames (Tom Hardy), and Yusuf (Dileep Rao). The score by Hans Zimmer is iconic, with that ticking sound.\n\nThemes include reality vs. dreams, memory, loss, and identity. The ending is ambiguous, with the spinning top falling or not. It\'s a 2010 film, 

In [15]:
# Structured output

model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', year=2010, rating=8.8, director='Christopher Nolan')

### Message output alongside Parsed structure

In [ ]:
# To get the Movie object alongside the raw response, you can set include_raw=True.
model_with_structure = model.with_structured_output(Movie, include_raw=True)

### Nested Structure

In [17]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    age: int = Field(description="The age of the actor")
    role: str = Field(description="The role of the actor in the movie")

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating of the movie")
    director: str = Field(description="The director of the movie")
    actors: list[Actor] = Field(description="The actors in the movie")
    genres: list[str] = Field(description="The genres of the movie")
    budget: float | None = Field(description="The budget of the movie in millions of dollars")

model_with_structure = model.with_structured_output(Movie)

response = model_with_structure.invoke("Provide the details about the movie Inception")
response

Movie(title='Inception', year=2010, rating=8.8, director='Christopher Nolan', actors=[Actor(name='Leonardo DiCaprio', age=45, role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', age=37, role='Arthur'), Actor(name='Elliot Page', age=27, role='Ariadne')], genres=['Sci-Fi', 'Action', 'Thriller'], budget=160.0)

### TypedDict

In [19]:
from typing_extensions import TypedDict

class Actor(TypedDict):
    name: str 
    age: int 
    role: str 

class Movie(TypedDict):
    title: str 
    year: int 
    rating: float 
    director: str 
    actors: list[Actor] 
    genres: list[str] 
    budget: float | None 

model_with_structure = model.with_structured_output(Movie)

response = model_with_structure.invoke("Provide the details about the movie Inception")
response

{'actors': [{'age': 37, 'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'age': 35, 'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'age': 28, 'name': 'Ellen Page', 'role': 'Caitlin'},
  {'age': 35, 'name': 'Tom Hardy', 'role': 'Fischer'}],
 'budget': 160000000,
 'director': 'Christopher Nolan',
 'genres': ['Action', 'Science Fiction', 'Thriller'],
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [21]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [ ]:
# This example uses create_agent, you can use init_chat_model as well

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str # The name of the contact
    email: str # The email of the contact
    phone: str # The phone number of the contact

agent = create_agent(
    model,
    response_format=ContactInfo,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Dow, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Dow, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='0381bae6-111a-415a-94ea-c5483e4d0fa1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract contact information from the given text: "John Dow, john@example.com, (555) 123-4567". \n\nFirst, I need to identify the different parts of the contact info. The name is John Dow. The email is clearly john@example.com. The phone number is (555) 123-4567, but I should format it without the parentheses and hyphens as per standard phone number formatting, maybe 555-123-4567 or just 5551234567. Wait, the function parameters don\'t specify formatting, so maybe I should keep it as provided.\n\nLooking at the function ContactInfo, the parameters required are name, email, and phone. All three are present here. The phone number includes parentheses and a space, but the function\'s parameters 

In [ ]:
# Pydantic using create_agent

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the contact")
    email: str = Field(description="The email of the contact")
    phone: str = Field(description="The phone number of the contact")

agent = create_agent(
    model,
    response_format=ContactInfo,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Dow, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Dow, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2b3bea97-92f3-4e62-82d1-be4f3561c8c0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given text: "John Dow, john@example.com, (555) 123-4567". Let me see. The function provided is called ContactInfo, and it requires name, email, and phone. \n\nFirst, I need to parse the name. The first part is "John Dow", which seems straightforward. The email is "john@example.com" – that\'s clearly an email address. The phone number is "(555) 123-4567". The function parameters require the phone number as a string. The user might expect the phone number to be formatted without the parentheses and hyphen, but the function doesn\'t specify any particular formatting, just a string. The example includes the phone as "(555) 123-4567", so maybe I should keep it as-is.\n\nW

In [6]:
result["structured_response"]

ContactInfo(name='John Dow', email='john@example.com', phone='(555) 123-4567')